# 🎙️ Malayalam Audiobook in YOUR Voice — IndicF5 (real cloning)

This notebook narrates a Malayalam PDF in a **cloned voice** from a short
reference clip, using AI4Bharat's **IndicF5** (the real thing — clones voice
*and* Malayalam dialect, unlike timbre-only tricks).

**Runtime → Change runtime type → GPU (T4)** before you start.

Steps: install → HF login → upload voice clip → auto-transcribe → upload PDF →
clone-narrate every chunk (resumable) → assemble a chaptered `.m4b` → download.


## 0. First, accept the model license (one time)
Open **https://huggingface.co/ai4bharat/IndicF5** in a new tab, sign in, and
click **Agree / Access repository**. Then create a **read** token at
**https://huggingface.co/settings/tokens**. You'll paste it two cells down.

In [ ]:
#@title 1. Install everything (one clean shot ~5 min, run once)
# Order matters: fix pkg_resources for Python 3.12 FIRST, then a matched stack.
import os
get_ipython().system('apt-get -qq install -y ffmpeg >/dev/null 2>&1')
get_ipython().system('pip install -q "setuptools==70.3.0"')
get_ipython().system('pip install -q git+https://github.com/ai4bharat/IndicF5.git')
get_ipython().system('pip install -q "torch==2.5.1" "torchaudio==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu124')
get_ipython().system('pip install -q "transformers==4.44.2" "numpy==1.26.4" pymupdf soundfile openai-whisper')
get_ipython().system('pip install -q "numpy==1.26.4" "setuptools==70.3.0"')  # re-assert LAST
print("Install done. GPU:", end=" ")
get_ipython().system('nvidia-smi --query-gpu=name --format=csv,noheader')
print("\n*** NOW: Runtime -> Restart session, then run cells 2 onward. Do NOT re-run this cell. ***")

## ⚠️ After Cell 1 finishes: Restart the session
Click **Runtime → Restart session** (this loads the freshly installed
libraries cleanly). Then continue with Cell 2 below. **Do not re-run Cell 1.**
Your uploaded files stay on disk; you only re-run cells 2–9.

In [ ]:
#@title 2. Log in to Hugging Face (paste your read token)
from huggingface_hub import login
login()  # paste the token from https://huggingface.co/settings/tokens

In [ ]:
#@title 3. Upload your voice clip (clean, 10-30s, one speaker)
from google.colab import files
import subprocess
up = files.upload()                 # pick a .wav/.m4a/.mp3
raw = list(up.keys())[0]
# -> 24kHz mono wav, lightly normalized
subprocess.run(["ffmpeg","-y","-i",raw,"-ac","1","-ar","24000",
                "-af","loudnorm=I=-20:TP=-2","ref.wav"], check=True)
print("saved ref.wav from", raw)

In [ ]:
#@title 4. Auto-transcribe the clip (Malayalam) -> ref_text
# openai-whisper on the GPU: no torchvision/ctranslate2 issues, accurate Malayalam.
get_ipython().system('pip -q install -U openai-whisper >/dev/null 2>&1')
import whisper
_wm = whisper.load_model("large-v3")
_res = _wm.transcribe("ref.wav", language="ml", fp16=True)
ref_text = _res["text"].strip()
print("AUTO-TRANSCRIPT (check it, edit in the next cell if wrong):\n")
print(ref_text)

In [ ]:
#@title 5. (Optional) Fix the transcript
# If the auto-transcript above is wrong, uncomment and paste the exact words:
# ref_text = "നിങ്ങളുടെ ക്ലിപ്പിലെ കൃത്യമായ മലയാളം ഇവിടെ"
print("Using ref_text:\n", ref_text)

In [ ]:
#@title 6. Upload the Malayalam PDF, then extract + chunk
from google.colab import files
import fitz, re, unicodedata
up = files.upload(); pdf_path = list(up.keys())[0]

_SENT=re.compile(r"([.?!।॥])"); _PG=re.compile(r"^\s*[\divxlcIVXLC]+\s*$")
def normalize(raw):
    t=unicodedata.normalize("NFC",raw)
    t="\n".join(l.rstrip() for l in t.split("\n") if not _PG.match(l))
    t=re.sub(r"(?<!\n)\n(?!\n)"," ",t); t=re.sub(r"[ \t]+"," ",t)
    return re.sub(r"\n{3,}","\n\n",t).strip()
def split_sent(t):
    p=_SENT.split(t); out=[];b=""
    for x in p:
        if _SENT.fullmatch(x): b+=x; out.append(b.strip()); b=""
        else: b+=x
    if b.strip(): out.append(b.strip())
    return [s for s in out if s]
def chunk(t,mx=350):
    ch=[];cur=""
    for s in split_sent(t):
        if len(s)>mx:
            if cur: ch.append(cur.strip()); cur=""
            w="" 
            for tok in re.split(r"(\s+|,|;)",s):
                if len(w)+len(tok)<=mx: w+=tok
                else:
                    if w.strip(): ch.append(w.strip())
                    w=tok
            if w.strip(): ch.append(w.strip())
        elif len(cur)+len(s)+1<=mx: cur=(cur+" "+s).strip()
        else:
            if cur: ch.append(cur.strip())
            cur=s
    if cur.strip(): ch.append(cur.strip())
    return ch

doc=fitz.open(pdf_path)
title=doc.metadata.get("title") or pdf_path.rsplit(".",1)[0]
full="\n\n".join(p.get_text("text") for p in doc if p.get_text("text").strip())
doc.close()
chunks=chunk(normalize(full))
print(f"{title}: {len(chunks)} chunks, {len(full):,} chars")

In [ ]:
#@title 7. Clone-narrate every chunk in your voice (resumable)
import os
os.environ.pop("TORCHDYNAMO_DISABLE", None)     # clear any stuck flag
import numpy as np, soundfile as sf, torch, torch._dynamo
torch._dynamo.reset()
torch._dynamo.config.disable = False            # compiler ON so checkpoint (_orig_mod) weights load
from transformers import AutoModel
from tqdm.auto import tqdm
model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True).to("cuda").eval()
torch._dynamo.config.disable = True             # OFF after load -> fast eager inference
os.makedirs("chunks", exist_ok=True)
for i, text in enumerate(tqdm(chunks), 1):
    out = f"chunks/c{i:05d}.wav"
    if os.path.exists(out):
        continue
    try:
        audio = np.asarray(model(text, ref_audio_path="ref.wav", ref_text=ref_text))
        if audio.dtype == np.int16 or np.abs(audio).max() > 1.5:
            audio = audio.astype(np.float32) / 32768.0
        sf.write(out, audio.astype(np.float32).squeeze(), 24000)
    except Exception as e:
        print("chunk", i, "failed:", e)
print("DONE - re-run to resume if interrupted")

In [ ]:
#@title 8. Assemble a chaptered .m4b
import os, subprocess, soundfile as sf, numpy as np
paths=[f"chunks/c{i:05d}.wav" for i in range(1,len(chunks)+1) if os.path.exists(f"chunks/c{i:05d}.wav")]
sr=sf.info(paths[0]).samplerate
sf.write("_sil.wav", np.zeros(int(sr*0.45),dtype=np.float32), sr)
sf.write("_sill.wav", np.zeros(int(sr*1.2),dtype=np.float32), sr)
lines=[]; t=0.0; cs=0.0; ci=1; chaps=[]
for k,cp in enumerate(paths):
    lines.append(f"file '{os.path.abspath(cp)}'"); t+=sf.info(cp).frames/sf.info(cp).samplerate
    last=k==len(paths)-1
    if (t-cs)>=20*60 or last:
        chaps.append((cs,t,f"Chapter {ci}")); ci+=1; cs=t+(1.2 if not last else 0)
        lines.append(f"file '{os.path.abspath("_sill.wav")}'"); t+=1.2 if not last else 0
    else:
        lines.append(f"file '{os.path.abspath("_sil.wav")}'"); t+=0.45
open("_c.txt","w").write("\n".join(lines))
subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i","_c.txt","-c:a","aac","-b:a","96k","_j.m4a"],check=True,capture_output=True)
meta=[";FFMETADATA1","title=Audiobook",""]
for s,e,n in chaps: meta+=["[CHAPTER]","TIMEBASE=1/1000",f"START={int(s*1000)}",f"END={int(e*1000)}",f"title={n}",""]
open("_m.txt","w").write("\n".join(meta))
subprocess.run(["ffmpeg","-y","-i","_j.m4a","-i","_m.txt","-map_metadata","1","-c","copy","audiobook.m4b"],check=True,capture_output=True)
print("audiobook.m4b ready —", len(chaps), "chapters")

In [ ]:
#@title 9. Download the audiobook
from google.colab import files
files.download("audiobook.m4b")